## 拓展：强制使用工具
### 1. tool_choice参数说明

bind_tools 可以传递参数tool_choice，用于控制是否强制使用工具。

- none：模型不会调用任何工具。
- auto：默认值，模型可以自主决定不调用或调用任意数量的工具。
- required ：模型必须调用工具，数量不限。

此外，tool_choice 还支持传递any，等价于required。

### 1.1 none值举例

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain.messages import HumanMessage
import os

#优先加载配置文件
load_dotenv(override=True)

#初始化大模型
model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

#定义工具
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取当天的天气

    Args:
        city: 城市名称

    Returns:
        当天指定城市的天气
    """
    return f"{city}今天晴朗"

model_with_tools = model_openai.bind_tools([get_weather], tool_choice="none")

messages = [
    HumanMessage(content="今天北京天气如何？别瞎编")
]

response = model_with_tools.invoke(messages)

response.pretty_print()

================================== Ai Message ==================================

由于我无法直接访问实时的互联网数据或气象卫星信息，因此无法为您提供**今天（实时）**北京的确切天气情况。

为了确保信息的准确性，避免“瞎编”，请您通过以下权威渠道查询最新天气：

1. **中国天气网**（www.weather.com.cn）
2. **中央气象台**官网或官方App
3. 手机自带的**天气应用**（通常接入的是国家气象局或国际权威气象机构数据）
4. 北京本地媒体如**北京日报**、**央视新闻**的气象播报

这些来源能提供最及时、准确的温度、降水概率、空气质量指数（AQI）等实时数据。建议您出行前务必查看最新版本预报。


### 1.2 auto值举例

In [31]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain.messages import HumanMessage
import os

#优先加载配置文件
load_dotenv(override=True)

#初始化大模型
model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

#定义工具
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取当天的天气

    Args:
        city: 城市名称

    Returns:
        当天指定城市的天气
    """
    return f"{city}今天晴朗"

model_with_tools = model_openai.bind_tools([get_weather], tool_choice="auto")

#注意：如果不问城市的天气，或者问得很模糊，就不会调用tool
messages = [
    HumanMessage(content="今天北京的天气如何？别瞎编")
]

response = model_with_tools.invoke(messages)

response.pretty_print()

================================== Ai Message ==================================

请问您想查询**哪个城市**的天气呢？告诉我具体城市名后，我会立即为您调取准确的实时天气数据。


### 1.3 required值举例

In [32]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain.messages import HumanMessage
import os

#优先加载配置文件
load_dotenv(override=True)

#初始化大模型
model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

#定义工具
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取当天的天气

    Args:
        city: 城市名称

    Returns:
        当天指定城市的天气
    """
    return f"{city}今天晴朗"

model_with_tools = model_openai.bind_tools([get_weather], tool_choice="required")
#注意：不能随便问，如果问的是和tool里完全无关的化，就不会调用tool。只有相关的问题
messages = [
    HumanMessage("我想出去玩，咋样")
]

response = model_with_tools.invoke(messages)

response.pretty_print()


================================== Ai Message ==================================
Tool Calls:
  get_weather (call_40c09b6ce60a4059882c174a)
 Call ID: call_40c09b6ce60a4059882c174a
  Args:
    city: 用户所在城市


### 1.4 强制调用特定的工具

In [33]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain.messages import HumanMessage
import os

#优先加载配置文件
load_dotenv(override=True)

#初始化大模型
model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
)

#定义工具
@tool(parse_docstring=True)
def get_weather1(city: str) -> str:
    """
    获取当天的天气

    Args:
        city: 城市名称

    Returns:
        当天指定城市的天气
    """
    return f"{city}今天晴朗"

#定义工具
@tool(parse_docstring=True)
def get_weather2(city: str) -> str:
    """
    获取当天的天气

    Args:
        city: 城市名称

    Returns:
        当天指定城市的天气
    """
    return f"{city}今天大雪"

model_with_tools = model_openai.bind_tools([get_weather1, get_weather2], tool_choice="get_weather2")

#注意：如果不问城市的天气，或者问得很模糊，就不会调用tool
messages = [
    HumanMessage(content="今天北京的天气如何？别瞎编")
]

response = model_with_tools.invoke(messages)

response.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_weather2 (call_c992f7b952cc419582b02eb1)
 Call ID: call_c992f7b952cc419582b02eb1
  Args:
    city: 北京
